In [49]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import ssl

In [50]:
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pranav/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [51]:
text = "This product is AMAZING!!! I love it 😍"
text = text.lower()
print("Lowercase :" ,text)

text = re.sub(r'[^\w\s!]', '', text)
print("No punctuation:",text)

words = text.split()
words = [w for w in words if w not in stopwords.words('english')]
text = " ".join(words)
print("Final Cleared:",text)

Lowercase : this product is amazing!!! i love it 😍
No punctuation: this product is amazing!!! i love it 
Final Cleared: product amazing!!! love


In [52]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s!]', '', text)
    words = text.split()
    words = [w for w in words if w not in stopwords.words('english')]
    return " ".join(words)


In [53]:
df = pd.read_csv("./data/fake-reviews.csv")
df = df[['text_', 'label']]
df = df.rename(columns={'text_': 'review'})
df = df.dropna()
df['clean_review'] = df['review'].apply(clean_text)
df[['review' , 'clean_review']].head()

,review,clean_review
0,"Love this! Well made, sturdy, and very comfor...",love this! well made sturdy comfortable love i...
1,"love it, a great upgrade from the original. I...",love great upgrade original ive mine couple years
2,This pillow saved my back. I love the look and...,pillow saved back love look feel pillow
3,"Missing information on how to use it, but it i...",missing information use great product price!
4,Very nice set. Good quality. We have had the s...,nice set good quality set two months


In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),   # bigrams 🔥
    min_df=2,
    max_df=0.9
)
X = vectorizer.fit_transform(df['clean_review'])
print(X.shape)

(40432, 20000)


In [80]:
from sklearn.model_selection import train_test_split

y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (32345, 20000)
Test size: (8087, 20000)


In [81]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)



,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [82]:
nb_pred = nb_model.predict(X_test)

In [83]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, nb_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, nb_pred))

Accuracy: 0.8916780017311735
Confusion Matrix:
 [[3515  501]
 [ 375 3696]]


In [84]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, nb_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, nb_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, nb_pred))

Accuracy: 0.8916780017311735
Confusion Matrix:
 [[3515  501]
 [ 375 3696]]

Classification Report:

              precision    recall  f1-score   support

          CG       0.90      0.88      0.89      4016
          OR       0.88      0.91      0.89      4071

    accuracy                           0.89      8087
   macro avg       0.89      0.89      0.89      8087
weighted avg       0.89      0.89      0.89      8087



In [85]:
def predict_review(text):
    text = clean_text(text)
    vector = vectorizer.transform([text])
    prediction = nb_model.predict(vector)
    return "Fake" if prediction[0] == "CG" else "Real"

print(predict_review("good and the  battery life is also good it lasts upto 7-10hrs🫶"))

Real


In [86]:
print(predict_review("Wow wow wow!!! This is the best product on the market!!! Everyone should buy this right now!!!"))

Real


In [87]:
print(predict_review("Wow wow wow!!! This is the best product on the market!!! Everyone should buy this right now!!!"))

Real


In [88]:
print(predict_review("This product exceeded all expectations and performs exceptionally well in all conditions highly recommended for all users"))

Real


In [89]:
print(predict_review("They are the perfect touch for me and the only thing I wish they had a little more space."))

Fake


In [96]:
import pickle

pickle.dump(nb_model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

In [75]:
import numpy as np

print("Unique predictions:", np.unique(nb_pred))

Unique predictions: ['CG' 'OR']


In [76]:
print(predict_review("Amazing amazing amazing!!! Best best best product ever ever ever!!!"))

Fake
